# CP2 Week 4 -- Data Quality Reports

**Course:** Computer Programming 2 (CP2)
**Prerequisites:** Weeks 1-3
**Focus:** missing analysis, outlier detection, quality scores

## Learning Objectives
- Generate comprehensive data quality reports
- Detect and characterize outliers using IQR
- Compute quality scores for your data
- Analyze missing value patterns
- Export quality reports as JSON

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: What Is a Data Quality Report?

A **data quality report** answers the question: "How good is this data?"

It summarizes:
- How many rows were dropped and why
- Which columns have missing values (and how many)
- Whether outliers exist and where
- An overall quality score

Think of it like a health checkup for your data.

### Data Quality Report Structure

```
+----------------------------------+
| DATA QUALITY REPORT              |
+----------------------------------+
| Dataset: 200 raw -> 180 clean    |
| Drop rate: 10%                   |
+----------------------------------+
| Missing Values:                  |
|   value: 5 (2.5%)                |
|   status: 3 (1.5%)               |
+----------------------------------+
| Outliers: 4 detected             |
|   IQR bounds: [12.3, 87.6]       |
+----------------------------------+
| Quality Score: 90/100            |
+----------------------------------+
```

In [ ]:
def generate_quality_report(raw_data, clean_data, drop_report, config):
    """Generate a comprehensive data quality report.
    
    Args:
        raw_data: original data before cleaning
        clean_data: data after cleaning
        drop_report: dict with "dropped" reasons from cleaning
        config: pipeline configuration
    
    Returns:
        dict suitable for JSON export
    """
    n_raw = len(raw_data)
    n_clean = len(clean_data)
    n_dropped = n_raw - n_clean
    
    report = {
        "project_name": config.get("project_name", "unknown"),
        "track": config.get("track", "unknown"),
        "version": config.get("version", "v2"),
        "dataset": {
            "n_raw": n_raw,
            "n_clean": n_clean,
            "n_dropped": n_dropped,
            "drop_rate_pct": round(n_dropped / n_raw * 100, 1) if n_raw > 0 else 0,
        },
        "cleaning_summary": drop_report.get("dropped", {}),
    }
    
    # Missing value analysis per column
    if raw_data:
        missing_per_col = {}
        for col in raw_data[0].keys():
            missing = sum(
                1 for row in raw_data
                if not row.get(col) or str(row[col]).strip() == ""
            )
            if missing > 0:
                missing_per_col[col] = {
                    "count": missing,
                    "pct": round(missing / n_raw * 100, 1),
                }
        report["missing_analysis"] = missing_per_col
    
    # Quality score (simple version)
    report["quality_score"] = round(n_clean / n_raw * 100, 1) if n_raw > 0 else 0
    
    return report

# Demo
raw = [
    {"a": "1", "b": "10"}, {"a": "2", "b": ""},
    {"a": "", "b": "30"},  {"a": "4", "b": "40"},
    {"a": "5", "b": "50"}, {"a": "6", "b": "abc"},
]
clean = [{"a": 1, "b": 10}, {"a": 4, "b": 40}, {"a": 5, "b": 50}]
drop_info = {"dropped": {"missing_a": 1, "missing_b": 1, "non_numeric_b": 1}}
config = {"project_name": "demo", "track": "data", "version": "v2"}

report = generate_quality_report(raw, clean, drop_info, config)

import json
print(json.dumps(report, indent=2))

**Expected Output:**
```
{
  "project_name": "demo",
  "track": "data",
  "version": "v2",
  "dataset": {
    "n_raw": 6,
    "n_clean": 3,
    "n_dropped": 3,
    "drop_rate_pct": 50.0
  },
  "cleaning_summary": {
    "missing_a": 1,
    "missing_b": 1,
    "non_numeric_b": 1
  },
  "missing_analysis": {
    "a": {"count": 1, "pct": 16.7},
    "b": {"count": 1, "pct": 16.7}
  },
  "quality_score": 50.0
}
```

---
## Part 2: Outlier Detection

An **outlier** is a value that is unusually far from the rest. The most common method to detect outliers is the **IQR (Interquartile Range)** method:

1. Sort the values
2. Find Q1 (25th percentile) and Q3 (75th percentile)
3. Compute IQR = Q3 - Q1
4. Any value below Q1 - 1.5*IQR or above Q3 + 1.5*IQR is an outlier

### IQR Outlier Detection

```
   outliers      normal range        outliers
   <------->  <----------------->  <--------->
   |         |         |         |           |
   Q1-1.5*IQR   Q1    median    Q3    Q3+1.5*IQR
   (lower)                            (upper)
```

In [ ]:
def detect_outliers(values, method="iqr"):
    """Detect outliers using the IQR method.
    
    Args:
        values: list of numbers
        method: detection method ("iqr")
    
    Returns:
        dict with bounds, outlier list, and summary
    """
    if not values or len(values) < 4:
        return {"outliers": [], "message": "Not enough data"}
    
    sorted_vals = sorted(values)
    n = len(sorted_vals)
    q1 = sorted_vals[n // 4]
    q3 = sorted_vals[3 * n // 4]
    iqr = q3 - q1
    
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    outliers = []
    for i, v in enumerate(values):
        if v < lower or v > upper:
            outliers.append({"index": i, "value": round(v, 2)})
    
    result = {
        "method": method,
        "q1": round(q1, 2),
        "q3": round(q3, 2),
        "iqr": round(iqr, 2),
        "lower_bound": round(lower, 2),
        "upper_bound": round(upper, 2),
        "n_outliers": len(outliers),
        "n_total": len(values),
        "outliers": outliers,
    }
    
    print("IQR Analysis:")
    print("  Q1=" + str(result["q1"]) + ", Q3=" + str(result["q3"])
          + ", IQR=" + str(result["iqr"]))
    print("  Bounds: [" + str(result["lower_bound"]) + ", "
          + str(result["upper_bound"]) + "]")
    print("  Outliers: " + str(result["n_outliers"])
          + " of " + str(result["n_total"]))
    
    return result

# Test with normal data + planted outliers
import random
random.seed(42)
values = [random.gauss(50, 10) for _ in range(50)]
values.extend([200, -50, 150])  # add outliers

result = detect_outliers(values)
print("\nOutlier values:")
for o in result["outliers"]:
    print("  Index " + str(o["index"]) + ": " + str(o["value"]))

**Expected Output:**
```
IQR Analysis:
  Q1=43.38, Q3=57.24, IQR=13.86
  Bounds: [22.59, 78.03]
  Outliers: 3 of 53

Outlier values:
  Index 50: 200
  Index 51: -50
  Index 52: 150
(values approximate due to random seed)
```

### Try It Yourself

Create a list of 20 exam scores (0-100) with one score of 5 and one of 99. Run outlier detection on it. Are those extreme scores flagged?

In [ ]:
# Try it: outlier detection on exam scores
# TODO: create scores and run detect_outliers


### Key Takeaway

- Data quality reports summarize the health of your data
- Always track missing values per column with counts and percentages
- IQR method: outlier if value < Q1-1.5*IQR or > Q3+1.5*IQR
- Quality score = percentage of rows that survived cleaning
- Export reports as JSON for reproducibility

---
## Mini-Quiz

In [ ]:
# Q1: What are the 4 main sections of a data quality report?
# Answer: 

# Q2: How does the IQR method detect outliers?
# Answer: 

# Q3: Why export quality reports as JSON?
# Answer: 

---
## Homework: 12 Exercises

### Review (1-4)

In [ ]:
# HW1: Generate a quality report for your project data.


In [ ]:
# HW2: Run outlier detection on your main numeric column.


In [ ]:
# HW3: What does a quality score of 75 mean?


In [ ]:
# HW4: Write missing_analysis for 3 columns of your data.


### Practice (5-8)

In [ ]:
# HW5: Add outlier info to your quality report.


In [ ]:
# HW6: Write a function that rates data quality as
# "excellent" (>95), "good" (>80), "fair" (>60), "poor" (<60).


In [ ]:
# HW7: Detect outliers using z-score method (alternative to IQR).
# z-score = (value - mean) / std; outlier if abs(z) > 3


In [ ]:
# HW8: Write a function that identifies columns with the most
# missing values and recommends whether to drop the column.


### Challenge (9-11)

In [ ]:
# HW9: Implement a "data quality dashboard" that prints a
# formatted text summary of all quality metrics.


In [ ]:
# HW10: Compare quality before and after cleaning -- show
# how cleaning improved data quality.


In [ ]:
# HW11: Write a function that automatically suggests cleaning
# rules based on the quality report.


### Mini-Project

In [ ]:
# HW12: Build a complete data quality module for your project.
# Requirements: missing analysis, outlier detection, quality score,
# formatted report, JSON export.


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)